# 02A | DataCo: data understanding, preparation and EDA

Inspect the historical order outcome and prediction-time feature availability; aggregate item rows into orders; visualize target balance, numerical distributions, correlation, shipping-mode association and training-period composition. Keep legitimate extremes; never infer disaster exposure from these associations.

No predictive models are trained here. This notebook processes only DataCo.


## Inputs and outputs

Complete notebook 01. Inputs are this experiment's public acquisition manifests and raw DataCo files, restored from Google Drive.

Outputs: Five DataCo chart panels; feature and leakage decisions; independently validated DataCo preparation. No tokenizer or classifier downloads.


## Configuration

Enable Drive, bootstrap and stage execution. Keep the experiment ID unchanged across the package. All actions default disabled.


In [ ]:
EXPERIMENT_ID = "colab-cpu-v3-clean"
DRIVE_FOLDER = "ChainShield-Colab"
ENABLE_DRIVE = False
BOOTSTRAP_EXPERIMENT = False
ALLOW_PACKAGE_INSTALL = False
ALLOW_DATA_DOWNLOADS = False
ALLOW_PRETRAINED_DOWNLOADS = False
RUN_STAGE = False
SYNC_INTERVAL_SECONDS = 60
VERIFY_PERSISTENCE_FULL = False
VERIFY_CHECKPOINT_PAIRS = False
CLEAR_STALE_WRITER_LOCK = False
CONFIRM_NO_OTHER_WRITER = False
DOWNLOAD_REPORT_ZIP = False

# Isolated plotting workers must not inherit the Colab inline backend.
if RUN_STAGE:
    import os
    os.environ["MPLBACKEND"] = "Agg"


## Shared definitions

Readable code is supplied once by its owning stage and persisted in Drive. Register these definitions before attaching; registration alone performs no preparation.


In [ ]:
# This small cell registers the visible stage code; it writes no files.
REGISTERED_FILES = {}
def register_stage_file(line, body):
    name=line.strip()
    if not name or name.startswith("/") or "\\" in name or ":" in name or ".." in name.split("/"):
        raise ValueError("Use a safe relative module name")
    REGISTERED_FILES[name]=body.rstrip("\n")+"\n"
get_ipython().register_magic_function(register_stage_file,"cell","chainshield_file")


### src/chainshield/data/dataco.py


In [ ]:
%%chainshield_file src/chainshield/data/dataco.py
"""Leakage-conscious order-level DataCo preparation; no fitted preprocessing."""
from __future__ import annotations
from collections import Counter, defaultdict
import csv
from datetime import datetime
from decimal import Decimal, InvalidOperation
from pathlib import Path


def assign_chronological_splits(rows, fractions):
    """Assign eligible orders using blocks of unique calendar dates."""
    dates = sorted({r["order_date"][:10] for r in rows if r["eligible"]})
    if len(dates) < 3:
        raise ValueError("At least three distinct eligible order dates are required")
    first = max(1, min(len(dates)-2, int(len(dates)*fractions["train"])))
    second = max(first+1, min(len(dates)-1, int(len(dates)*(fractions["train"]+fractions["dev"]))))
    date_split = {day: "train" if i < first else "dev" if i < second else "test"
                  for i, day in enumerate(dates)}
    for row in rows:
        row["split"] = date_split[row["order_date"][:10]] if row["eligible"] else ""
    return {"date_blocks": {"train_end": dates[first-1], "dev_start": dates[first],
             "dev_end": dates[second-1], "test_start": dates[second]},
            "distinct_dates": len(dates)}


def aggregate_orders(source_rows, spec):
    """Retain targets for evaluation but expose an explicit feature allowlist."""
    groups = defaultdict(list)
    source_count = 0
    for line, row in enumerate(source_rows, 2):
        source_count += 1
        order_id = str(row.get("Order Id") or "").strip()
        groups[order_id or f"MISSING_ORDER_ID_ROW_{line}"].append(row)
    result = []
    for order_id, items in sorted(groups.items()):
        reasons = []
        if order_id.startswith("MISSING_ORDER_ID_ROW_"):
            reasons.append("missing_order_id")
        labels = {str(r.get(spec["target"]) or "").strip() for r in items}
        label = next(iter(labels)) if len(labels) == 1 else ""
        if len(labels) != 1:
            reasons.append("conflicting_target")
        elif label not in {"0", "1"}:
            reasons.append("invalid_target")
        item_ids = [str(r.get("Order Item Id") or "").strip() for r in items]
        if any(not value for value in item_ids):
            reasons.append("missing_order_item_id")
        if len(set(item_ids)) != len(item_ids):
            reasons.append("duplicate_order_item_id")
        record = {"order_id": order_id, "label": label, "item_count": len(items)}
        dates = {str(r.get("order date (DateOrders)") or "").strip() for r in items}
        date = ""
        if len(dates) != 1:
            reasons.append("conflicting_order_date")
        else:
            try:
                date = datetime.strptime(next(iter(dates)), "%m/%d/%Y %H:%M").isoformat()
            except ValueError:
                reasons.append("invalid_order_date")
        record["order_date"] = date
        record["order_month"] = int(date[5:7]) if date else ""
        record["order_weekday"] = datetime.fromisoformat(date).weekday() if date else ""
        stable = dict(spec["categorical_fields"])
        stable[spec["scheduled_field"]] = "scheduled_shipping_days"
        for original, output in stable.items():
            values = {str(r.get(original) or "").strip() for r in items}
            record[output] = next(iter(values)) if len(values) == 1 else ""
            if len(values) > 1:
                reasons.append("conflicting_feature:" + output)
            if not record[output]:
                reasons.append("missing_feature:" + output)
        try:
            scheduled = Decimal(record["scheduled_shipping_days"])
            if not scheduled.is_finite() or scheduled < 0:
                raise InvalidOperation
            record["scheduled_shipping_days"] = str(scheduled)
        except (InvalidOperation, ValueError):
            reasons.append("invalid_scheduled_shipping_days")
        for original, output in spec["sum_fields"].items():
            try:
                values = [Decimal(str(r.get(original) or "")) for r in items]
                if not all(v.is_finite() and v >= 0 for v in values):
                    raise InvalidOperation
                record[output] = str(sum(values, Decimal(0)))
            except (InvalidOperation, ValueError):
                record[output] = ""
                reasons.append("invalid_numeric:" + output)
        for original, output in spec["distinct_fields"].items():
            record[output] = len({str(r.get(original) or "").strip() for r in items
                                  if str(r.get(original) or "").strip()})
        record["eligible"] = not reasons
        record["exclusion_reasons"] = "|".join(sorted(set(reasons)))
        result.append(record)
    return result, source_count


def prepare(raw_root: Path, config):
    spec = config["dataco"]
    path = raw_root / "DataCoSupplyChainDataset.csv"
    with path.open(encoding=spec["encoding"], newline="") as handle:
        reader = csv.DictReader(handle)
        required = {"Order Id", "Order Item Id", "order date (DateOrders)", spec["target"],
                    spec["scheduled_field"], *spec["categorical_fields"],
                    *spec["sum_fields"], *spec["distinct_fields"]}
        if not required.issubset(reader.fieldnames or []):
            raise ValueError("DataCo required fields are missing")
        rows, source_count = aggregate_orders(reader, spec)
    boundaries = assign_chronological_splits(rows, config["splits"])
    manifest = [{"record_id": r["order_id"], "task": "shipment_late_risk",
                 "group_id": r["order_id"], "split": r["split"],
                 "label": r["label"], "eligible": r["eligible"],
                 "exclusion_reasons": r["exclusion_reasons"]} for r in rows]
    features = [key for key in rows[0] if key not in
                {"order_id","label","order_date","eligible","exclusion_reasons","split"}]
    quality = {
        "source_rows": source_count, "canonical_orders": len(rows),
        "eligible_orders": sum(r["eligible"] for r in rows),
        "excluded_orders": sum(not r["eligible"] for r in rows),
        "exclusion_reason_counts": dict(Counter(reason for r in rows
            for reason in r["exclusion_reasons"].split("|") if reason)),
        "order_level_label_counts": dict(Counter(r["label"] for r in rows)),
        "eligible_split_counts": dict(Counter(r["split"] for r in rows if r["eligible"])),
        "split_label_counts": {s: dict(Counter(r["label"] for r in rows
            if r["eligible"] and r["split"] == s)) for s in ("train","dev","test")},
        "feature_columns": features, "identifier_columns": ["order_id"],
        "excluded_predictors": spec["excluded_predictors"],
        "prediction_cutoff": spec["prediction_cutoff"], **boundaries,
        "limitations": ["Feature availability at order creation is a documented modeling assumption.",
            "No scaling, imputation, encoding or feature learning fitted on any partition.",
            "No verified disaster linkage or final-mile outcome is established.",
            "Customer credential, identity and street fields are excluded from canonical outputs.",
            "Eventual model training must use feature_columns only; label and IDs are not predictors."],
    }
    return {"canonical": rows, "split_manifest": manifest, "quality": quality,
            "label_map": {"shipment_late_risk": {"0": "no_recorded_late_risk", "1": "recorded_late_risk"}}}


### configs/preprocessing/dataco.json


In [ ]:
%%chainshield_file configs/preprocessing/dataco.json
{
  "schema_version": 1,
  "pipeline_version": "chainshield-preparation-v1",
  "seed": 42,
  "splits": {
    "train": 0.7,
    "dev": 0.15,
    "test": 0.15
  },
  "normalization": "HTML unescape, Unicode NFC, whitespace collapse; casefold only for exact-text keys",
  "dataco": {
    "encoding": "cp1252",
    "prediction_cutoff": "order_creation",
    "target": "Late_delivery_risk",
    "categorical_fields": {
      "Shipping Mode": "shipping_mode",
      "Type": "payment_type",
      "Customer Segment": "customer_segment",
      "Market": "market",
      "Order Country": "order_country",
      "Order Region": "order_region",
      "Order State": "order_state"
    },
    "scheduled_field": "Days for shipment (scheduled)",
    "sum_fields": {
      "Order Item Quantity": "total_quantity",
      "Order Item Total": "order_total",
      "Order Item Discount": "total_discount",
      "Sales": "gross_sales"
    },
    "distinct_fields": {
      "Category Id": "category_count",
      "Department Id": "department_count",
      "Order Item Cardprod Id": "product_count"
    },
    "excluded_predictors": [
      "Days for shipping (real)",
      "Delivery Status",
      "Order Status",
      "shipping date (DateOrders)",
      "Late_delivery_risk",
      "Benefit per order",
      "Order Profit Per Order",
      "Order Item Profit Ratio",
      "Sales per customer",
      "Customer Email",
      "Customer Password",
      "Customer Fname",
      "Customer Lname",
      "Customer Street",
      "Customer Id",
      "Order Customer Id",
      "Latitude",
      "Longitude"
    ],
    "split_policy": "Chronological blocks of unique order calendar dates; no calendar date spans partitions; conflicting orders quarantined"
  }
}


### configs/eda/dataco.json


In [ ]:
%%chainshield_file configs/eda/dataco.json
{
  "ml": {
    "numeric_features": [
      "item_count",
      "order_month",
      "order_weekday",
      "scheduled_shipping_days",
      "total_quantity",
      "order_total",
      "total_discount",
      "gross_sales",
      "category_count",
      "department_count",
      "product_count"
    ],
    "categorical_features": [
      "shipping_mode",
      "payment_type",
      "customer_segment",
      "market",
      "order_country",
      "order_region",
      "order_state"
    ]
  }
}


### src/chainshield/research/eda_dataco.py


In [ ]:
%%chainshield_file src/chainshield/research/eda_dataco.py
from collections import Counter,defaultdict
import math,statistics,re,json
from .eda import truth,numeric,describe,label_counts,correlation

def dataco_profile(canonical,feature_spec):
    train=[r for r in canonical if r["split"]=="train" and truth(r["eligible"])]
    if not train:raise ValueError("No eligible DataCo training orders")
    numbers={name:[numeric(r.get(name)) for r in train] for name in feature_spec["numeric_features"]}
    numeric_summary={name:dict(describe(values),missing_or_invalid=len(values)-sum(x is not None for x in values))
                     for name,values in numbers.items()}
    categorical={};rates=[]
    for name in feature_spec["categorical_features"]:
        groups=defaultdict(list)
        for row in train:groups[row.get(name) or "[MISSING]"].append(int(row["label"]))
        categorical[name]={"cardinality":len(groups),
            "top_counts":[{"value":value,"count":len(labels)} for value,labels in
                          sorted(groups.items(),key=lambda pair:(-len(pair[1]),pair[0]))[:20]]}
        for value,labels in sorted(groups.items()):
            rates.append({"feature":name,"value":value,"orders":len(labels),"late_orders":sum(labels),
                          "late_rate":sum(labels)/len(labels),"small_support":len(labels)<30})
    relationships=[]
    names=list(numbers)
    for index,name in enumerate(names):
        for other in names[index+1:]:
            relationships.append({"feature_a":name,"feature_b":other,**correlation(numbers[name],numbers[other])})
    periods=defaultdict(list)
    for row in train:periods[row["order_date"][:7]].append(int(row["label"]))
    return {"partition":"train","unit":"order","records":len(train),
            "class_balance":label_counts(r["label"] for r in train),
            "numeric_summary":numeric_summary,"categorical_summary":categorical,"categorical_late_rates":rates,
            "numeric_correlations":relationships,
            "monthly":[{"month":key,"orders":len(value),"late_rate":sum(value)/len(value)} for key,value in sorted(periods.items())],
            "interpretation":"Training associations and descriptive outliers do not establish causal effects or justify automatic feature/row removal."}

def add_profiles(profile,canonical,prepared,tokenizer,include_examples,root,inherited,config):
    profile["datasets"]["dataco"]=dataco_profile(canonical["dataco"],config["ml"])


### src/chainshield/research/plots_dataco.py


In [ ]:
%%chainshield_file src/chainshield/research/plots_dataco.py
import math
from .eda import rows,truth

def render_panels(data,root,save,plt,np):
    d=data["datasets"]["dataco"]
    fig,ax=plt.subplots(figsize=(8,4));counts=d["class_balance"]
    bars=ax.bar(["On time / other (0)","Recorded late risk (1)"],[r["count"] for r in counts],color=["#2b8cbe","#e58b45"])
    ax.bar_label(bars,labels=[f'{r["count"]:,}\n{r["percent"]:.2f}%' for r in counts],padding=3)
    ax.set_ylim(0,max(r["count"] for r in counts)*1.2);ax.set_ylabel("Training orders");ax.set_title(f'DataCo target distribution | n={d["records"]:,}')
    save(fig,"01_dataco_balance","DataCo target balance",
         f'The training target is {counts[1]["percent"]:.2f}% positive.',
         "Keep the natural class distribution. Class weighting is a candidate decision, not an automatic requirement.")
    prepared=(root/data.get("prepared_root","data/processed")).resolve()
    prepared.relative_to(root)
    orders=[r for r in rows(prepared/"dataco/canonical.csv") if r["split"]=="train" and truth(r["eligible"])]
    names=list(d["numeric_summary"]);fig,axes=plt.subplots(3,4,figsize=(16,10))
    for ax,name in zip(axes.flat,names):
        values=[float(r[name]) for r in orders if str(r.get(name,"")).strip()]
        ax.hist(values,bins=30,color="#2b8cbe",edgecolor="white")
        ax.set_title(name.replace("_"," "));ax.set_ylabel("Training orders")
        ax.axvline(d["numeric_summary"][name]["median"],color="#d95f0e",linestyle="--",linewidth=1)
    for ax in list(axes.flat)[len(names):]:ax.axis("off")
    fig.suptitle(f'DataCo numerical distributions | n={len(orders):,}; dashed line = median',fontsize=15)
    save(fig,"02_dataco_distributions","Eligible feature distributions",
         "Distributions are measured after canonical eligibility checks; they do not imply the raw dataset had no missing values.",
         "Retain legitimate extremes initially. Evaluate any transformation as a documented train-only candidate.")
    matrix=np.eye(len(names))
    for pair in d["numeric_correlations"]:
        i,j=names.index(pair["feature_a"]),names.index(pair["feature_b"])
        matrix[i,j]=matrix[j,i]=pair["pearson_r"] if pair["pearson_r"] is not None else np.nan
    fig,ax=plt.subplots(figsize=(12,10));im=ax.imshow(matrix,vmin=-1,vmax=1,cmap="coolwarm")
    ax.set_xticks(range(len(names)),[n.replace("_"," ") for n in names],rotation=65,ha="right")
    ax.set_yticks(range(len(names)),[n.replace("_"," ") for n in names])
    for i in range(len(names)):
        for j in range(len(names)):
            if math.isfinite(matrix[i,j]) and abs(matrix[i,j])>.7:ax.text(j,i,f"{matrix[i,j]:.3f}",ha="center",va="center",fontsize=8,color="black")
    fig.colorbar(im,ax=ax,label="Training Pearson correlation")
    ax.set_title(f'DataCo numerical relationships | n={d["records"]:,}')
    save(fig,"03_dataco_correlations","Correlated feature groups",
         "Financial totals and product/category counts have very strong correlations.",
         "Run bounded feature-group ablations. Correlation alone does not prove a feature is useless or establish causality.")
    modes=[r for r in d["categorical_late_rates"] if r["feature"]=="shipping_mode"]
    fig,ax=plt.subplots(figsize=(9,4));bars=ax.bar([r["value"] for r in modes],[100*r["late_rate"] for r in modes],color="#4878a8")
    ax.bar_label(bars,labels=[f'{100*r["late_rate"]:.1f}%\nn={r["orders"]:,}' for r in modes],padding=3)
    ax.set_ylim(0,115);ax.set_ylabel("Recorded late-risk percentage");ax.set_title("DataCo training association by shipping mode")
    save(fig,"04_shipping_mode","Shipping-mode association",
         "Recorded target rates vary substantially by mode and its promised schedule.",
         "Use shipping-only, non-shipping and full-feature comparisons. These are associations, not evidence that changing mode causes a benefit.")
    fig,axes=plt.subplots(2,1,figsize=(13,6),sharex=True);monthly=d["monthly"];x=list(range(len(monthly)))
    axes[0].bar(x,[r["orders"] for r in monthly],color="#2b8cbe");axes[0].set_ylabel("Training orders")
    axes[1].plot(x,[100*r["late_rate"] for r in monthly],marker="o",color="#d95f0e")
    axes[1].set_ylabel("Late-risk %");axes[1].set_ylim(0,100)
    axes[1].set_xticks(x,[r["month"] for r in monthly],rotation=70,ha="right");axes[0].set_title("DataCo training-period volume and recorded risk")
    save(fig,"05_dataco_time","Training-period composition",
         "These observations cover only the historical training period.",
         "Preserve chronological evaluation; do not infer disaster exposure or seasonality from these lines alone.")


In [ ]:
import hashlib,json
EXPECTED_STAGE_FILES={'src/chainshield/data/dataco.py': 'b835285086feb6e683284ae8c2518639f1293e15e77613e586fcb1a9239743e9', 'configs/preprocessing/dataco.json': 'ad5edf5ff0fe9a0dc6feeaf96f1cb56a719eeb036bd2f992ec8674edac18f8fd', 'configs/eda/dataco.json': 'b92e433ee33d37ddeb0ef80b0e25428e50ea13ee7b8195bff034ed3871c1fd72', 'src/chainshield/research/eda_dataco.py': '10501753e9fc416580fafbe3c237567bdcad7f330e7ee25bb7c463ba8aa13e75', 'src/chainshield/research/plots_dataco.py': 'bbc35c462a8ff9a47bb6220cbeb804035adaee9d441975c2a0b8b03248063c91'}
assert set(REGISTERED_FILES)==set(EXPECTED_STAGE_FILES)
for name,expected in EXPECTED_STAGE_FILES.items():
    if hashlib.sha256(REGISTERED_FILES[name].encode()).hexdigest()!=expected:raise ValueError("Changed source: "+name)


## Attach and verify prerequisites


In [ ]:
# Connect the Colab experiment and load the shared runtime.
from pathlib import Path
CONTROLLER_SHA = '8c092c59baffc0804aeca86b4f0ac3fb6d4be3cd96eea96c9e7299e559d9c9e6'
IMPLEMENTATION_SHA256 = CONTROLLER_SHA
rt = None
if BOOTSTRAP_EXPERIMENT:
    if not ENABLE_DRIVE:
        raise PermissionError("Set ENABLE_DRIVE=True")
    from google.colab import drive
    drive.mount("/content/drive")
    runtime_path=Path("/content/drive/MyDrive")/DRIVE_FOLDER/"experiments"/EXPERIMENT_ID/"workspace/.replication/controller.py"
    if not runtime_path.is_file():
        raise RuntimeError("Complete Version 3 notebook 00 first; its shared controller is missing")
    runtime_code=runtime_path.read_text(encoding="utf-8")
    if hashlib.sha256(runtime_code.encode()).hexdigest()!=CONTROLLER_SHA:
        raise ValueError("Shared runtime version differs; do not mix notebook packages")
    exec(compile(runtime_code,"<Colab shared runtime>","exec"),globals())
    rt=Replica(EXPERIMENT_ID,drive_base=DRIVE_FOLDER,allow_drive=True)
    if CLEAR_STALE_WRITER_LOCK:
        rt.clear_stale_lock(acknowledge_no_other_writer=CONFIRM_NO_OTHER_WRITER)
else:
    print("Bootstrap disabled: no Drive access or filesystem changes")


In [ ]:
if RUN_STAGE:
    if rt is None:raise RuntimeError("Enable Drive/bootstrap first")
    rt.require_stage('01')
    rt.install_sources(REGISTERED_FILES)
    rt.environment(allow_install=ALLOW_PACKAGE_INSTALL)
else:
    print("Stage execution disabled")


## Data understanding and preparation choices

Inspect the historical order outcome and prediction-time feature availability; aggregate item rows into orders; visualize target balance, numerical distributions, correlation, shipping-mode association and training-period composition. Keep legitimate extremes; never infer disaster exposure from these associations.

Preparation verifies public input hashes and expected semantic/test-membership digests. All analysis below uses eligible training records; final-test labels never select settings.


In [ ]:
if RUN_STAGE:
    rt.execute("02A-prepare",["scripts/dataset_research_workflow.py","prepare","--dataset","dataco"],SYNC_INTERVAL_SECONDS)
else:
    print("Stage execution disabled")


## Measured EDA, interpretation and decisions

Read the observation and preparation implication for every chart. Coverage and correlation do not prove a model will perform better.


In [ ]:
if RUN_STAGE:
    rt.execute("02A-eda",["scripts/dataset_research_workflow.py","eda","--dataset","dataco"],SYNC_INTERVAL_SECONDS)
else:
    print("Stage execution disabled")


In [ ]:
if RUN_STAGE:
    print(json.dumps(read_json(rt.root/"data/processed/"/"dataco"/"quality.json"),indent=2))
else:
    print("Stage execution disabled")


In [ ]:
if RUN_STAGE:
    from IPython.display import display,Image,Markdown
    for variant in ('control',):
        folder=rt.root/"artifacts/research-v2/eda"/("dataco_"+variant)
        display(Markdown("### DataCo | "+variant))
        display(Markdown((folder/"EDA_Review.md").read_text()))
        for image in sorted((folder/"figures_v2").glob("*.png")):display(Image(filename=str(image)))
        summary=read_json(folder/"dataset-profiles.json")
        print("Preparation safeguards:",json.dumps(summary["preparation_controls"],indent=2))
else:
    print("Stage execution disabled")


## Persist and complete

Finish only after preparation and charts succeed. A restart of the Python kernel alone may retain partial scratch files; use a clean temporary Colab runtime for archive restoration when needed. Preserve Drive evidence.


In [ ]:
if RUN_STAGE:
    rt.finish_stage("02A")
else:
    print("Stage execution disabled")
